# Components — Transmission Grouping & Coastdown Signal Experiment

**Question:** Do strict structured transmission candidate groups carry repeatable out-of-sample information about whole-vehicle coastdown force?

This is a read-only scientific experiment. A positive signal is not a transmission-loss decomposition, and this notebook never writes to a database.

## 1. Portable paths and experiment execution

In [ ]:
from pathlib import Path
import os
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in search_roots if (path / "AGENTS.md").exists() and (path / "etl").exists())
DB_PATH = REPO_ROOT / "data/db/staging/eco_drive_canonical_candidate.db"
OUTPUT_DIR = REPO_ROOT / "artifacts/components/transmission_experiment"
assert DB_PATH.exists(), DB_PATH

from etl.scripts.sprint_12_transmission_grouping_signal_experiment import run_experiment

n_permutations = int(os.environ.get("TRANSMISSION_EXPERIMENT_PERMUTATIONS", "500"))
result = run_experiment(DB_PATH, OUTPUT_DIR, n_permutations=n_permutations, build_notebook_artifact=False)
result


## 2. Data / transmission-field audit

In [ ]:
field_audit = pd.read_csv(OUTPUT_DIR / 'TRANSMISSION_FIELD_AUDIT.csv')
field_audit

## 3. Duplicate and carryover control

In [ ]:
sample_audit = pd.read_csv(OUTPUT_DIR / 'EXPERIMENT_SAMPLE_AUDIT.csv')
sample_audit[["included_excluded", "exclusion_reason"]].value_counts(dropna=False)


## 4. Candidate transmission grouping and coverage

In [ ]:
groups = pd.read_csv(OUTPUT_DIR / 'TRANSMISSION_CANDIDATE_GROUPS.csv')
groups[["identity_status", "usable_ge3", "usable_ge5"]].value_counts().sort_index()


In [ ]:
groups.sort_values(["n_unique_models", "n_vdes"], ascending=False).head(20)


## 5. ABC → force curves (outcome domain only)

In [ ]:
curves = pd.read_csv(OUTPUT_DIR / 'FORCE_CURVE_EXPERIMENT_LONG.csv')
example_ids = curves["vde_id"].drop_duplicates().head(8)
example_curves = curves[curves["vde_id"].isin(example_ids)].pivot(index="speed_kph", columns="vde_id", values="observed_force_N")
if plt is None:
    print("matplotlib is unavailable; displaying the complete chart-driving table instead.")
    print(example_curves)
else:
    ax = example_curves.plot(figsize=(8, 4), alpha=0.7)
    ax.set(title="Canonical whole-vehicle coastdown force curves", xlabel="Speed (km/h)", ylabel="Force (N)")
    plt.show()


## 6. Baseline nuisance model

Model 0 uses mass, year, make, category, drive architecture, electrification, and polynomial speed interactions. It excludes Target-ABC-derived CdA/RRC and all component/decomposition estimates.

In [ ]:
model_comparison = pd.read_csv(OUTPUT_DIR / 'MODEL_COMPARISON.csv')
model_comparison

## 7. Transmission-group augmented model and grouped validation

Model 1 adds only a candidate-group constant and candidate-group × speed term. It does not fit a group quadratic term or publish transmission ABC.

In [ ]:
validation = pd.read_csv(OUTPUT_DIR / 'GROUP_VALIDATION.csv')
validation[validation["scope_type"].isin(["OVERALL", "FOLD", "SPEED_KPH"])]


## 8. Residual-curve similarity and permutation test

In [ ]:
permutations = pd.read_csv(OUTPUT_DIR / 'PERMUTATION_RESULTS.csv')
observed = permutations.iloc[0]
if plt is None:
    print("matplotlib is unavailable; displaying permutation quantiles instead.")
    print(permutations.iloc[1:]["statistic_value"].describe(percentiles=[0.025, 0.5, 0.975]))
else:
    ax = permutations.iloc[1:]["statistic_value"].plot.hist(bins=30, alpha=0.75, figsize=(7, 4))
    ax.axvline(observed["statistic_value"], color="red", linewidth=2, label="Observed")
    ax.set(title="Stratified label-permutation null", xlabel="Within-group residual-curve RMSE (N)")
    ax.legend()
    plt.show()
observed


## 9. Sensitivity / robustness

In [ ]:
sensitivity = pd.read_csv(OUTPUT_DIR / 'SENSITIVITY_RESULTS.csv')
sensitivity

## 10. Representative BMW groups

In [ ]:
bmw = groups[
    groups["make"].astype(str).str.upper().eq("BMW")
    & groups["identity_status"].eq("STRICT_CANDIDATE")
    & groups["usable_ge3"].eq("YES")
].sort_values(["n_unique_models", "n_vdes"], ascending=False)
bmw.head(15)


## 11. Failure cases, confounders, and decision

- The source provides no explicit transmission hardware/model code, so all useful groups are candidates rather than proven same-part groups.
- Independent CdA and tire/RRC evidence is unavailable at useful coverage; their variation remains nuisance/confounding.
- The result is about reproducible grouping signal in whole-vehicle force curves, never causal transmission neutral drag.
- Read the generated Sprint result report for the prespecified decision and exact metrics.


In [ ]:
print((REPO_ROOT / 'docs/sprints/SPRINT_12_TRANSMISSION_GROUPING_EXPERIMENT_RESULT.md').read_text(encoding='utf-8'))